In [11]:
import pandas as pd

# Load the prepared dataset and normalize the time column.
df = pd.read_csv("gold_1m.csv")
df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
df = df.dropna(subset=["time"]).sort_values("time").reset_index(drop=True)


In [8]:
df.head()

,time,open,high,low,close,volume,rsi
0,2024-01-02 05:01:00+00:00,2062.14,2069.79,2062.14,2069.79,4,NaN
1,2024-01-02 05:02:00+00:00,2069.79,2069.79,2069.49,2069.53,66,NaN
2,2024-01-02 05:03:00+00:00,2069.53,2069.55,2069.39,2069.44,95,NaN
3,2024-01-02 05:04:00+00:00,2069.44,2069.67,2069.40,2069.64,93,NaN
4,2024-01-02 05:05:00+00:00,2069.64,2069.90,2069.56,2069.89,75,NaN


In [16]:
df = df[15:]
df.head()

,time,open,high,low,close,volume,rsi,target,year,month,day,dayofweek,hour,minute,minute_of_day,minute_sin,minute_cos,prediction
15,2024-01-02 05:16:00+00:00,2071.46,2071.46,2071.05,2071.11,91,45.709571,-0.02,2024,1,2,1,5,16,316,0.981627,0.190809,NaN
16,2024-01-02 05:17:00+00:00,2071.11,2071.16,2070.75,2071.09,116,45.432474,0.31,2024,1,2,1,5,17,317,0.982450,0.186524,NaN
17,2024-01-02 05:18:00+00:00,2071.09,2071.41,2071.09,2071.40,78,50.446804,-0.47,2024,1,2,1,5,18,318,0.983255,0.182236,NaN
18,2024-01-02 05:19:00+00:00,2071.40,2071.53,2070.91,2070.93,116,43.865360,-0.06,2024,1,2,1,5,19,319,0.984041,0.177944,NaN
19,2024-01-02 05:20:00+00:00,2070.93,2071.03,2070.75,2070.87,86,43.092454,0.01,2024,1,2,1,5,20,320,0.984808,0.173648,NaN


In [12]:
# Build target: next close minus current close.
df["target"] = df["close"].shift(-1) - df["close"]
df = df.dropna(subset=["target"]).reset_index(drop=True)

# Time features for the model. Raw datetime is expanded into numeric inputs.
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day
df["dayofweek"] = df["time"].dt.dayofweek
df["hour"] = df["time"].dt.hour
df["minute"] = df["time"].dt.minute
df["minute_of_day"] = df["hour"] * 60 + df["minute"]
df["minute_sin"] = np.sin(2 * np.pi * df["minute_of_day"] / 1440)
df["minute_cos"] = np.cos(2 * np.pi * df["minute_of_day"] / 1440)

df.head()


,time,open,high,low,close,volume,rsi,target,year,month,day,dayofweek,hour,minute,minute_of_day,minute_sin,minute_cos
0,2024-01-02 05:01:00+00:00,2062.14,2069.79,2062.14,2069.79,4,NaN,-0.26,2024,1,2,1,5,1,301,0.967046,0.254602
1,2024-01-02 05:02:00+00:00,2069.79,2069.79,2069.49,2069.53,66,NaN,-0.09,2024,1,2,1,5,2,302,0.968148,0.250380
2,2024-01-02 05:03:00+00:00,2069.53,2069.55,2069.39,2069.44,95,NaN,0.20,2024,1,2,1,5,3,303,0.969231,0.246153
3,2024-01-02 05:04:00+00:00,2069.44,2069.67,2069.40,2069.64,93,NaN,0.25,2024,1,2,1,5,4,304,0.970296,0.241922
4,2024-01-02 05:05:00+00:00,2069.64,2069.90,2069.56,2069.89,75,NaN,0.01,2024,1,2,1,5,5,305,0.971342,0.237686


In [17]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

feature_columns = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "rsi",
    "year",
    "month",
    "day",
    "dayofweek",
    "hour",
    "minute",
    "minute_of_day",
    "minute_sin",
    "minute_cos",
]

X = df[feature_columns]
y = df["target"]

# Chronological split to avoid leakage in time series data.
split_index = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

model = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Train rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"RMSE: {rmse:.6f}")
print(f"MAE: {mae:.6f}")
print(f"R2: {r2:.6f}")


Train rows: 542323
Test rows: 135581
RMSE: 1.464153
MAE: 0.950334
R2: -0.265070


In [14]:
# Save the trained model and prediction output for later use.
df.loc[X_test.index, "prediction"] = predictions
df.to_csv("gold_1m_xgboost_features.csv", index=False)
model.save_model("xgboost_xauusd.json")
